In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2

try:
    from PIL import ImageFile

    ImageFile.LOAD_TRUNCATED_IMAGES = True
except Exception:
    pass

torch.manual_seed(3407)
if torch.cuda.is_available():
    torch.cuda.manual_seed(3407)

cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
if not os.path.isdir(test_dir):
    raise FileNotFoundError(f"test_dir not found: {test_dir}")

top_level_jpgs = [
    f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))
]
nested_dir = os.path.join(test_dir, "test_images")
if len(top_level_jpgs) == 0 and os.path.isdir(nested_dir):
    test_dir = nested_dir

train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
train_img_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images/"
if not os.path.isdir(train_img_dir):
    raise FileNotFoundError(f"train_img_dir not found: {train_img_dir}")

img_size = 384
batch_size = 16
num_workers = min(8, (os.cpu_count() or 8))
num_classes = 5
tta = False

# Change (score): enable short head-only fine-tuning; same loop/optimizer/loss, but no longer skipped.
# This is the smallest legitimate change to move accuracy upward from near-random toward the target.
train_epochs = 1
train_lr = 3e-4
weight_decay = 0.01
freeze_backbone = True  # train only head for speed/stability

torch.set_num_threads(min(8, (os.cpu_count() or 8)))
torch.set_num_interop_threads(1)

from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
vit_model = vit_b_16(weights=weights)
vit_model.heads.head = torch.nn.Linear(vit_model.heads.head.in_features, num_classes)

vit_model.to(device)
vit_model = vit_model.to(memory_format=torch.channels_last)

if hasattr(torch, "compile"):
    try:
        vit_model = torch.compile(vit_model, mode="reduce-overhead", fullgraph=False)
    except Exception as e:
        print("torch.compile unavailable; continuing in eager mode:", repr(e))




cuda


Downloading: "https://download.pytorch.org/models/vit_b_16_swag-9ac1b537.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16_swag-9ac1b537.pth
100%|██████████| 331M/331M [00:03<00:00, 115MB/s]


In [2]:
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Args:
        data_dir: base directory to the images.
        transform: set of transforms to be used.
        ttas: list of transforms for test-time augmentation.
    """

    def __init__(self, data_dir, transform=None, ttas=None, img_size=384):
        super().__init__(root=data_dir)
        self.transform = transform
        self.ttas = ttas

        exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
        files = []
        with os.scandir(data_dir) as it:
            for entry in it:
                if entry.is_file():
                    n = entry.name
                    if n.lower().endswith(exts):
                        files.append(n)
        self.images = sorted(files)

        if len(self.images) == 0:
            raise RuntimeError(f"No image files found in: {data_dir}")

    def __getitem__(self, idx):
        filename = self.images[idx]
        img = Image.open(os.path.join(self.root, filename)).convert("RGB")

        if self.ttas is not None and self.transform is not None:
            img = [self.transform(t(img)) for t in self.ttas]
        elif self.transform:
            img = self.transform(img)

        return img, filename

    def __len__(self):
        return len(self.images)


class CassavaTrainDataset(VisionDataset):
    def __init__(self, df, data_dir, transform=None):
        super().__init__(root=data_dir)
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self._image_ids = self.df["image_id"].to_numpy()
        self._labels = self.df["label"].to_numpy()

    def __getitem__(self, idx):
        filename = self._image_ids[idx]
        label = int(self._labels[idx])
        img_path = os.path.join(self.root, filename)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.df)




In [3]:
# Change (score): use the pretrained ViT weights' recommended preprocessing.
# This preserves resize/crop/normalize semantics while aligning exactly to the weights' expectations.
vit_preprocess = weights.transforms()

test_transforms = vit_preprocess
train_transforms = vit_preprocess

if tta:
    ttas = [
        v2.RandomResizedCrop((img_size, img_size), (0.5, 1)),
        v2.RandomRotation(180),
        v2.RandomVerticalFlip(1),
        v2.RandomAffine(180),
        v2.RandomPerspective(p=1),
    ]
else:
    ttas = None

test_dataset = CassavaDataset(test_dir, transform=test_transforms, ttas=ttas)

g = torch.Generator()
g.manual_seed(3407)


def _seed_worker(worker_id: int):
    seed = 3407 + worker_id
    torch.manual_seed(seed)


_loader_kwargs = dict(
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    prefetch_factor=2 if num_workers > 0 else None,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,  # keep deterministic ordering
    drop_last=False,
    worker_init_fn=_seed_worker if num_workers > 0 else None,
    generator=g,
    **{k: v for k, v in _loader_kwargs.items() if v is not None},
)

normalizer = torch.nn.Softmax(dim=1)

train_df = pd.read_csv(train_csv_path)
train_dataset = CassavaTrainDataset(train_df, train_img_dir, transform=train_transforms)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=False,
    worker_init_fn=_seed_worker if num_workers > 0 else None,
    generator=g,
    **{k: v for k, v in _loader_kwargs.items() if v is not None},
)




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [4]:
# Change (score): previously skipped training caused a random 5-class head => near-random accuracy.
# Enable the existing training loop for 1 epoch head-only fine-tuning to learn cassava labels quickly.
SKIP_TRAINING_TO_MEET_TIMEOUT = False

if not SKIP_TRAINING_TO_MEET_TIMEOUT:
    vit_model.train()

    if freeze_backbone:
        for p in vit_model.parameters():
            p.requires_grad = False
        for p in vit_model.heads.parameters():
            p.requires_grad = True

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        (p for p in vit_model.parameters() if p.requires_grad),
        lr=train_lr,
        weight_decay=weight_decay,
    )

    for epoch in range(train_epochs):
        running_loss = 0.0
        n_seen = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True).contiguous(
                memory_format=torch.channels_last
            )
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = vit_model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            n_seen += bs

        print(
            f"epoch {epoch+1}/{train_epochs} - train_loss: {running_loss / max(1, n_seen):.4f}"
        )




/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


epoch 1/1 - train_loss: 0.6131


In [5]:
all_names = []
all_preds = []

vit_model.eval()

if torch.cuda.is_available():
    cudnn.benchmark = True

with torch.inference_mode():
    for inputs, filenames in test_loader:
        if tta:
            inputs = (
                torch.cat(inputs, dim=0)
                .to(device, non_blocking=True)
                .contiguous(memory_format=torch.channels_last)
            )
            fnames = filenames

            preds = normalizer(vit_model(inputs))
            batch_preds = torch.stack(torch.split(preds, len(fnames)), dim=0)
            mean_preds = torch.mean(batch_preds, dim=0)
            pred_labels = torch.argmax(mean_preds, 1).tolist()
        else:
            inputs = inputs.to(device, non_blocking=True).contiguous(
                memory_format=torch.channels_last
            )
            fnames = filenames

            preds = normalizer(vit_model(inputs))
            pred_labels = torch.argmax(preds, 1).tolist()

        all_names.extend(fnames)
        all_preds.extend(pred_labels)




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [6]:
sample_path = "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"
sample_submission = pd.read_csv(sample_path)

pred_map = dict(zip(all_names, all_preds))
missing = [
    img_id
    for img_id in sample_submission["image_id"].tolist()
    if img_id not in pred_map
]
if missing:
    for img_id in missing:
        pred_map[img_id] = 0

my_submission = sample_submission.copy()
my_submission["label"] = my_submission["image_id"].map(pred_map).astype(int)

assert len(my_submission) == len(sample_submission)
assert list(my_submission.columns) == ["image_id", "label"]

my_submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", my_submission.shape)




Wrote submission.csv with shape: (2676, 2)


In [7]:
my_submission.head()

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,1
3,1234555380.jpg,1
4,1234571117.jpg,3
